# baseline (H100 수정본)

이 베이스라인 코드는 `사전학습 모델 로드`, `배치 학습`, `파인튜닝`, `양자화`, `PEFT` 등이 적용된 버전입니다.

**H100 80GB GPU / Google Colab Pro 환경 기준으로 수정된 버전입니다.**

# 환경 준비

개발 환경에 필요한 라이브러리 버전을 고정하고 설치합니다.

- 아래 셀 실행
- 런타임 재시작 후 다시 실행
- hello 출력 시 torch 설치 완료

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!unzip -q "/content/drive/MyDrive/data.zip" -d "/content"

In [3]:
# [수정] 버전 고정 설치로 재현 가능한 환경 구성
# flash-attn은 torch/CUDA 준비 확인 후 마지막에 설치
!pip -q install \
    "transformers>=4.50.0,<5.0.0" \
    "accelerate>=1.0.0" \
    "peft>=0.14.0" \
    "bitsandbytes>=0.45.0" \
    "datasets>=2.19.0" \
    pillow pandas

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 88.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 23.9 MB/s eta 0:00:00


In [1]:
import torch

print(torch.__version__)
print("CUDA 사용 가능:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name())
print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")

2.10.0+cu128
CUDA 사용 가능: True
GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition
VRAM: 102.0 GB


In [ ]:
# [수정] torch/CUDA 확인 후 flash-attn 설치 (H100은 sm_90 아키텍처)
import torch
assert torch.cuda.is_available(), "CUDA 없이는 flash-attn 설치 불가"
# !pip install flash-attn --no-build-isolation

# 데이터 준비

개발에 필요한 데이터를 준비합니다.

- train.csv, train 폴더
- test.csv, test 폴더
- sample_submission.csv

# 라이브러리, 데이터, 설정

In [ ]:
import transformers
print(transformers.__version__)

In [ ]:
# ===== 기본 환경 설정 및 데이터 로드 =====
import os, math, random
import numpy as np
import pandas as pd
import transformers
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from dataclasses import dataclass
import torch
from typing import Any
from transformers import (
    AutoModelForImageTextToText,
    AutoProcessor,
    BitsAndBytesConfig,
    get_linear_schedule_with_warmup
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from tqdm import tqdm

print(transformers.__version__)

Image.MAX_IMAGE_PIXELS = None

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

MODEL_ID = "Qwen/Qwen3-VL-32B-Instruct"  # H100 80GB에서 4bit 양자화 시 문제없음
IMAGE_SIZE = 384
SEED = 42

# [수정] numpy seed 추가 및 재현성 강화
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

DATA_ROOT = "/content"

DEBUG = False
DEBUG_N = 300

USE_VALID = False
VAL_RATIO = 0.1
USE_DEV_PSEUDO = False

# [수정] 출력 디렉토리 사전 생성 (학습 루프 내 중복 생성 제거)
os.makedirs("./outputs/qwen3_32b_lora", exist_ok=True)
os.makedirs("./outputs", exist_ok=True)

train_df = pd.read_csv(os.path.join(DATA_ROOT, "train.csv"))
test_df  = pd.read_csv(os.path.join(DATA_ROOT, "test.csv"))

# [수정] dev.csv 존재 여부 방어 처리
dev_csv_path = os.path.join(DATA_ROOT, "dev.csv")
dev_df = pd.read_csv(dev_csv_path) if os.path.exists(dev_csv_path) else pd.DataFrame()

if DEBUG:
    train_df = train_df.sample(n=min(DEBUG_N, len(train_df)), random_state=SEED).reset_index(drop=True)

print(f"train: {len(train_df)}, test: {len(test_df)}, dev: {len(dev_df)}")

# 모델, Processor

H100 80GB 환경에서 Qwen3-VL-32B 4bit 양자화 시 약 30~44GB 소모.

#### 실습 참고 내용

    챕터 5-1 PEFT(파라미터 효율적 튜닝)
    - LoRA 구현 : LoraConfig()

In [ ]:
# ===== dev 데이터를 다수결 기반 pseudo label로 활용 (선택 옵션) =====
def maybe_get_dev_pseudo(df):
    candidate_cols = [c for c in df.columns if "응답" in c]

    out_rows = []
    for _, row in df.iterrows():
        votes = []
        for c in candidate_cols:
            v = str(row[c]).strip().lower()
            if v in ["a", "b", "c", "d"]:
                votes.append(v)

        if not votes:
            continue

        cnt = pd.Series(votes).value_counts()
        if cnt.iloc[0] >= 3:
            out_rows.append({
                "id": row["id"],
                "path": row["path"],
                "question": row["question"],
                "a": row["a"],
                "b": row["b"],
                "c": row["c"],
                "d": row["d"],
                "answer": cnt.index[0]
            })

    return pd.DataFrame(out_rows)


if USE_DEV_PSEUDO and not dev_df.empty:
    pseudo = maybe_get_dev_pseudo(dev_df)
    train_df = pd.concat([train_df, pseudo], ignore_index=True)

In [ ]:
# ===== 4bit 양자화 + LoRA 기반 VLM 모델 로드 =====
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,  # H100 bfloat16 최적화
)

processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    min_pixels=IMAGE_SIZE * IMAGE_SIZE,
    max_pixels=IMAGE_SIZE * IMAGE_SIZE,
    trust_remote_code=True,
)
processor.tokenizer.padding_side = "left"

base_model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    attn_implementation="flash_attention_2",  # H100 필수 최적화
    device_map={"": 0},
    trust_remote_code=True,
)

# [수정] prepare_model_for_kbit_training이 gradient_checkpointing을 내부에서 처리
# 기존: base_model.gradient_checkpointing_enable() 먼저 호출 → 순서 역전 버그
base_model = prepare_model_for_kbit_training(
    base_model,
    use_gradient_checkpointing=True  # 내부에서 enable 처리
)

# [수정] Flash Attention 2 실제 적용 여부 확인
from transformers.utils import is_flash_attn_2_available
print("Flash Attention 2 사용 가능:", is_flash_attn_2_available())

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    task_type="CAUSAL_LM",
)

model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

# 프롬프트 템플릿

#### 실습 참고 내용

    챕터 5-1 PEFT(파라미터 효율적 튜닝)
    - 프롬프트 템플릿 : convert_to_chatml(), formatting_prompts_func()

In [ ]:
# ===== 문제를 단순한 4지선다 형태로 입력하는 프롬프트 =====
SYSTEM_INSTRUCT = "Choose correct answer: a, b, c, or d."

CHOICE_KEYS = ["a","b","c","d"]

def build_prompt(row):
    return (
        f"{row['question']}\n"
        f"(a) {row['a']}\n"
        f"(b) {row['b']}\n"
        f"(c) {row['c']}\n"
        f"(d) {row['d']}\n\n"
        f"Answer:"
    )

# Custom Dataset, Collator

#### 실습 참고 내용

    챕터 1-2 MLP 구현
    - TensorDataset()

    챕터 5-2 데이터 생성 및 파인튜닝 (향후 학습 분량)
    - IntentDataset()

In [ ]:
# ===== 이미지 + 텍스트를 묶어 모델 입력 형태로 변환 =====
def resolve_path(path):
    return path if os.path.isabs(path) else os.path.join(DATA_ROOT, path)

class VQADataset(Dataset):
    def __init__(self, df, mode="train"):
        self.df = df.reset_index(drop=True)
        self.mode = mode

    def __len__(self):
        return len(self.df)

    def __getitem__(self, i):
        row = self.df.iloc[i]
        img = Image.open(resolve_path(row["path"])).convert("RGB")

        prompt = build_prompt(row)

        messages = [
            {"role":"system","content":[{"type":"text","text":SYSTEM_INSTRUCT}]},
            {"role":"user","content":[
                {"type":"image","image":img},
                {"type":"text","text":prompt}
            ]}
        ]

        item = {"messages":messages, "image":img}

        if self.mode=="train":
            item["gold"] = row["answer"]

        return item

In [ ]:
# ===== 정답 토큰(a/b/c/d) 위치만 학습하도록 loss 마스킹 =====
@dataclass
class AnswerOnlyCollator:
    processor: Any
    train: bool = True

    def __call__(self, batch):
        prompt_texts, full_texts, images = [], [], []

        for s in batch:
            prompt = self.processor.apply_chat_template(
                s["messages"],
                tokenize=False,
                add_generation_prompt=True
            )
            prompt_texts.append(prompt)
            images.append(s["image"])

            if self.train:
                full_texts.append(prompt + str(s["gold"]).strip().lower())

        enc = self.processor(
            text=full_texts if self.train else prompt_texts,
            images=images,
            padding=True,
            return_tensors="pt"
        )

        if not self.train:
            return enc

        labels = torch.full_like(enc["input_ids"], -100)
        seq_len = enc["input_ids"].shape[1]

        # [수정] left padding을 고려한 실제 정답 토큰 위치 계산
        # 기존: answer_pos = seq_len - 1 (항상 맨 끝 — padding 무시한 버그)
        for i in range(len(batch)):
            valid_len = int(enc["attention_mask"][i].sum().item())
            start_idx = seq_len - valid_len       # padding 이후 실제 시작 위치
            answer_pos = start_idx + valid_len - 1  # 실제 마지막 유효 토큰 위치
            labels[i, answer_pos] = enc["input_ids"][i, answer_pos]

        enc["labels"] = labels
        return enc

# DataLoader

#### 실습 참고 내용

    챕터 3-1 Transfer Learning 기반의 CNN 모델 학습
    - 데이터로더 정의 : DataLoader()

In [ ]:
# ===== 학습용 데이터 로더 구성 =====
# [수정] 외부 shuffle 제거 — DataLoader의 shuffle=True가 담당
# 기존: train_df = train_df.sample(frac=1, ...) → DataLoader shuffle과 이중 처리
train_df = train_df.reset_index(drop=True)

train_ds = VQADataset(train_df, "train")

train_loader = DataLoader(
    train_ds,
    batch_size=2,        # H100 80GB에서 여유 있으면 4로 증가 가능
    num_workers=2,       # [수정] Colab 환경은 2 이하 권장 (4 → 2)
    shuffle=True,
    collate_fn=AnswerOnlyCollator(processor, True),
    pin_memory=True,
    prefetch_factor=2,
)

# fine-tuning

#### 실습 참고 내용

    챕터 1-2 MLP 구현
    - 모델 정의 : SimpleMLP(), SequentialMLP()

    챕터 3-1 Transfer Learning 기반의 CNN 모델 학습
    - 학습 루프 : 문제 6: 모델 학습을 위한 반복문
    - 추론 : with torch.no_grad(), model.eval()

In [ ]:
# ===== LoRA 파인튜닝 학습 루프 =====
GRAD_ACCUM = 8  # 실질 배치 사이즈: 2 * 8 = 16
EPOCHS = 1
LR = 1e-4

optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
num_training_steps = EPOCHS * math.ceil(len(train_loader) / GRAD_ACCUM)
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    int(num_training_steps * 0.03),
    num_training_steps
)

for epoch in range(EPOCHS):
    model.train()
    running = 0.0
    # [수정] 실제 누적 step 수 추적 (running loss 분모 오류 수정)
    steps_since_update = 0
    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1} [train]", unit="batch")

    optimizer.zero_grad(set_to_none=True)

    for step, batch in enumerate(progress_bar, start=1):
        # [수정] Tensor만 device로 이동 (non-Tensor 항목 필터링)
        batch = {k: v.to(device) for k, v in batch.items() if isinstance(v, torch.Tensor)}

        out = model(**batch)
        loss = out.loss / GRAD_ACCUM

        loss.backward()
        running += loss.item()
        steps_since_update += 1

        if step % GRAD_ACCUM == 0 or step == len(train_loader):
            # [수정] gradient clipping 추가 (학습 안정성)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad(set_to_none=True)

            # [수정] 실제 누적된 step 수로 평균 계산
            # 기존: min(GRAD_ACCUM, step) — step이 전체 누적 step이라 분모 오류
            avg_loss = running / steps_since_update
            progress_bar.set_postfix({"loss": f"{avg_loss:.4f}"})
            running = 0.0
            steps_since_update = 0

# [수정] import os 중복 선언 제거 (상단에서 이미 import)
# [수정] os.makedirs는 Cell 10에서 사전 생성했으므로 중복 제거
model.save_pretrained("./outputs/qwen3_32b_lora")
processor.save_pretrained("./outputs/qwen3_32b_lora")

# [수정] Google Drive 백업 추가 (Colab 런타임 종료 시 로컬 파일 소실 방지)
import shutil
drive_save_path = "/content/drive/MyDrive/outputs/qwen3_32b_lora"
if os.path.exists("/content/drive/MyDrive"):
    shutil.copytree("./outputs/qwen3_32b_lora", drive_save_path, dirs_exist_ok=True)
    print(f"✅ 모델 학습 완료 및 Drive 백업 성공: {drive_save_path}")
else:
    print("✅ 모델 학습 완료 (Drive 미연결 — 로컬 저장만 완료)")

# inference

#### 실습 참고 내용

    챕터4-1 RAG 기반 Customer Service AI 에이전트 개발
    - 데이터 파서 : langchain_core.output_parsers(), StrOutputParser()

    챕터 3-1 Transfer Learning 기반의 CNN 모델 학습
    - 추론 : with torch.no_grad(), model.eval()

In [ ]:
# ===== generate 대신 a/b/c/d logits 비교로 직접 선택 =====
choice_ids = {}
for c in CHOICE_KEYS:
    ids = processor.tokenizer.encode(c, add_special_tokens=False)
    if len(ids) != 1:
        raise ValueError(f"{c} is not a single token: {ids}")
    choice_ids[c] = ids[0]

model.eval()
preds = []

test_ds = VQADataset(test_df, "test")
test_loader = DataLoader(
    test_ds,
    batch_size=4,        # [수정] 8 → 4 (32B 모델 추론 안정성 확보)
    num_workers=2,       # [수정] 4 → 2 (Colab 환경)
    shuffle=False,
    collate_fn=AnswerOnlyCollator(processor, False)
)

with torch.no_grad():
    for batch in tqdm(test_loader, desc="Inference", unit="batch"):
        batch = {k: v.to(device) for k, v in batch.items() if isinstance(v, torch.Tensor)}

        with torch.amp.autocast("cuda", dtype=torch.bfloat16, enabled=torch.cuda.is_available()):
            out = model(**batch)

        logits = out.logits
        row_idx = torch.arange(logits.size(0), device=device)

        # left padding이면 마지막 입력 토큰은 항상 맨 끝
        last_pos = torch.full(
            (logits.size(0),),
            logits.size(1) - 1,
            dtype=torch.long,
            device=device
        )

        next_logits = logits[row_idx, last_pos, :]

        choice_logits = torch.stack(
            [next_logits[:, choice_ids[c]] for c in CHOICE_KEYS],
            dim=1
        )

        pred_idx = choice_logits.argmax(dim=1).tolist()
        preds.extend([CHOICE_KEYS[i] for i in pred_idx])

In [ ]:
# 제출 파일 생성
submission = pd.DataFrame({"id": test_df["id"], "answer": preds})
submission.to_csv("./outputs/submission.csv", index=False)
print("Saved outputs/submission.csv")

In [ ]:
# 모델 응답 예시
print(preds[:10])